Carregando as bibliotecas

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import re #biblioteca para manipulação de strings

from sklearn.feature_extraction.text import TfidfVectorizer #Pacote para extração de tokens
from sklearn.metrics.pairwise import linear_kernel #Pacote para calcular a similaridade via cosseno

Importando o arquivo de vendas "movies_ratings.csv"

In [2]:
dataRaw = pd.read_csv(r"C:\Users\celso\OneDrive\Área de Trabalho\trabalhos\netflix_titles.csv")
dataRaw.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
1,80117401,Movie,Jandino: Whatever it Takes,NaN,Jandino Asporaat,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,Stand-Up Comedy,Jandino Asporaat riffs on the challenges of ra...
2,70234439,TV Show,Transformers Prime,NaN,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids' TV,"With the help of three human allies, the Autob..."
3,80058654,TV Show,Transformers: Robots in Disguise,NaN,"Will Friedle, Darren Criss, Constance Zimmer, ...",United States,"September 8, 2018",2016,TV-Y7,1 Season,Kids' TV,When a prison ship crash unleashes hundreds of...
4,80125979,Movie,#realityhigh,Fernando Lebrija,"Nesta Cooper, Kate Walsh, John Michael Higgins...",United States,"September 8, 2017",2017,TV-14,99 min,Comedies,When nerdy high schooler Dani finally attracts...


In [3]:
dataRaw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6234 entries, 0 to 6233
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       6234 non-null   int64 
 1   type          6234 non-null   object
 2   title         6234 non-null   object
 3   director      4265 non-null   object
 4   cast          5664 non-null   object
 5   country       5758 non-null   object
 6   date_added    6223 non-null   object
 7   release_year  6234 non-null   int64 
 8   rating        6224 non-null   object
 9   duration      6234 non-null   object
 10  listed_in     6234 non-null   object
 11  description   6234 non-null   object
dtypes: int64(2), object(10)
memory usage: 584.6+ KB


Remover todos os NAN's das colunas.

In [4]:
dataRaw.dropna(subset=['cast','title','description','listed_in'],inplace=True,axis=0)
dataRaw = dataRaw.reset_index(drop=True)

dataRaw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5664 entries, 0 to 5663
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       5664 non-null   int64 
 1   type          5664 non-null   object
 2   title         5664 non-null   object
 3   director      3909 non-null   object
 4   cast          5664 non-null   object
 5   country       5271 non-null   object
 6   date_added    5654 non-null   object
 7   release_year  5664 non-null   int64 
 8   rating        5657 non-null   object
 9   duration      5664 non-null   object
 10  listed_in     5664 non-null   object
 11  description   5664 non-null   object
dtypes: int64(2), object(10)
memory usage: 531.1+ KB


Neste exercícios iremos utilizar os seguintes campos para busca de similaridade:

- type: tipo de mídia (filme, tv show, etc.)
- título: nome da mídia
- listed_in: categoria onde é apresentado.
- description: informação texto livre sobre a mídia.

Para aplicar o modelo de recomendação vamos limpar os textos e combiná-los.

In [5]:
dataRaw['listed_in'] = [re.sub(r'[^\w\s]', '', t) for t in dataRaw['listed_in']]
dataRaw['cast'] = [re.sub(',',' ',re.sub(' ','',t)) for t in dataRaw['cast']]
dataRaw['description'] = [re.sub(r'[^\w\s]', '', t) for t in dataRaw['description']]
dataRaw['title'] = [re.sub(r'[^\w\s]', '', t) for t in dataRaw['title']]

Agora vamos combinar os campos textuais numa única coluna chamanda "combined".

In [6]:
dataRaw["combined"] = dataRaw['listed_in'] + '  ' + dataRaw['cast'] + ' ' + dataRaw['title'] + ' ' + dataRaw['description']
dataRaw.drop(['listed_in','cast','description'],axis=1,inplace=True)
dataRaw.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,combined
0,81145628,Movie,Norm of the North King Sized Adventure,"Richard Finn, Tim Maltby","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,Children Family Movies Comedies AlanMarriott...
1,80117401,Movie,Jandino Whatever it Takes,NaN,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,StandUp Comedy JandinoAsporaat Jandino Whatev...
2,70234439,TV Show,Transformers Prime,NaN,United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids TV PeterCullen SumaleeMontano FrankWelke...
3,80058654,TV Show,Transformers Robots in Disguise,NaN,United States,"September 8, 2018",2016,TV-Y7,1 Season,Kids TV WillFriedle DarrenCriss ConstanceZimm...
4,80125979,Movie,realityhigh,Fernando Lebrija,United States,"September 8, 2017",2017,TV-14,99 min,Comedies NestaCooper KateWalsh JohnMichaelHig...


Tokenizaremos a coluna "combined" utilizando o método TF-IDF.

In [7]:
vectorizer = TfidfVectorizer(stop_words='english')
matrix = vectorizer.fit_transform(dataRaw["combined"])
matrix.shape

(5664, 46919)

Calcular a similaridade por cosseno.

In [8]:
cosine_similarities = linear_kernel(matrix,matrix)
cosine_similarities.shape

(5664, 5664)

Criar um índice para os filmes.

In [9]:
indices = pd.Series(dataRaw.index, index=dataRaw['title']).drop_duplicates()
indices

title
Norm of the North King Sized Adventure           0
Jandino Whatever it Takes                        1
Transformers Prime                               2
Transformers Robots in Disguise                  3
realityhigh                                      4
                                              ... 
Kikoriki                                      5659
Red vs Blue                                   5660
Maron                                         5661
A Young Doctors Notebook and Other Stories    5662
Friends                                       5663
Length: 5664, dtype: int64

Criar uma função para retornar itens semelhantes.

In [10]:
def get_similar(title, indices, cosine_similarities, num_recommend = 10):

  idx = indices[title]

  # Obtem todas os pares de scores de similaridade de todos os filmes com o filme alvo
  sim_scores = list(enumerate(cosine_similarities[idx]))

  # Ordena os filmes com base no score de similaridade
  sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

  # Obtem o score dos num_recommend filmes mais proximos
  top_similar = sim_scores[1:num_recommend+1]

  # Obtem o indice dos filmes
  movie_indices = [i[0] for i in top_similar]

  # Retorna os num_recommend filmes mais proximos
  return movie_indices

In [11]:
idx = get_similar("Naruto", indices, cosine_similarities, 10)

In [12]:
indices[idx]

C:\Users\celso\AppData\Local\Temp\ipykernel_26924\282710547.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  indices[idx]


title
Naruto Shippûden the Movie Bonds                              643
Naruto Shippuden The Movie                                    644
Naruto Shippuden  Blood Prison                                335
Naruto the Movie 2 Legend of the Stone of Gelel               338
Naruto Shippûden the Movie The Will of Fire                   336
Naruto the Movie 3 Guardians of the Crescent Moon Kingdom     339
Naruto Shippuden The Movie The Lost Tower                     337
Naruto the Movie Ninja Clash in the Land of Snow              340
Saint Seiya The Lost Canvas                                  1600
Beyblade Metal Fusion                                        4112
dtype: int64

In [ ]:
def RecommendMovies(user_ratings, indices, cosine_sim, df, num_recommend=10):
  
    movies = user_ratings["movie"]
    ratings = user_ratings["rating"]
    
    sim_scores_total = {}

    for movie, rating in zip(movies, ratings):
        if movie in indices:
            idx = indices[movie]
            sim_scores = list(enumerate(cosine_similarities[idx]))
            for i, score in sim_scores:
                if i not in sim_scores_total:
                    sim_scores_total[i] = 0
                sim_scores_total[i] += score * rating

    # Ordena os filmes com base na pontuação acumulada
    sorted_scores = sorted(sim_scores_total.items(), key=lambda x: x[1], reverse=True)

    # Gera as recomendações, excluindo os filmes que o usuário já avaliou
    recommended_movies = []
    for idx, _ in sorted_scores:
        movie_title = df['title'].iloc[idx]
        if movie_title not in movies:
            recommended_movies.append(movie_title)
        if len(recommended_movies) == num_recommend:
            break

    return recommended_movies

In [ ]:
usuario = {
    "movie": ["Inception", "The Matrix", "Interstellar"],
    "rating": [5, 4, 5]
}

recomendacoes = RecommendMovies(usuario, indices, cosine_similarities, dataRaw)

print("Recomendações de filmes para você:")
for i, filme in enumerate(recomendacoes, 1):
    print(f"{i}. {filme}")

Recomendações de filmes para você:
1. The Matrix Reloaded
2. The Matrix Revolutions
3. Transcendence
4. 9
5. Brick
6. Black Mirror Bandersnatch
7. Æon Flux
8. Dragonheart
9. Arès
10. Apollo 18
